In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout,LSTM,GRU,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [46]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [47]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [48]:
df_train

,Datetime,PJME_MW,year,month,hour,day_num,day_of_month,week_of_year,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24
0,2002-01-09 01:00:00,0.306289,0.0,0.000000,0.043478,0.333333,0.266667,0.019231,0,0.345497,0.313937,0.286042,0.456556,0.234076
1,2002-01-09 02:00:00,0.286738,0.0,0.000000,0.086957,0.333333,0.266667,0.019231,0,0.306289,0.297609,0.271632,0.456097,0.236377
2,2002-01-09 03:00:00,0.279890,0.0,0.000000,0.130435,0.333333,0.266667,0.019231,0,0.286738,0.291394,0.268766,0.455444,0.240153
3,2002-01-09 04:00:00,0.278416,0.0,0.000000,0.173913,0.333333,0.266667,0.019231,0,0.279890,0.294912,0.273654,0.454754,0.244299
4,2002-01-09 05:00:00,0.289982,0.0,0.000000,0.217391,0.333333,0.266667,0.019231,0,0.278416,0.310060,0.292026,0.453764,0.250095
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101635,2013-08-13 20:00:00,0.560813,1.0,0.636364,0.869565,0.166667,0.400000,0.615385,0,0.592205,0.599199,0.419973,0.532663,0.430098
101636,2013-08-13 21:00:00,0.558074,1.0,0.636364,0.913043,0.166667,0.400000,0.615385,0,0.560813,0.590393,0.431118,0.530359,0.422228
101637,2013-08-13 22:00:00,0.517455,1.0,0.636364,0.956522,0.166667,0.400000,0.615385,0,0.558074,0.551796,0.412262,0.528420,0.415776
101638,2013-08-13 23:00:00,0.449342,1.0,0.636364,1.000000,0.166667,0.400000,0.615385,0,0.517455,0.477594,0.363194,0.526359,0.411341


In [49]:
df_test

,Datetime,PJME_MW,year,month,hour,day_num,day_of_month,week_of_year,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24
0,2016-02-07 13:00:00,0.327568,1.272727,0.090909,0.565217,1.000000,0.200000,0.076923,1,0.336248,0.337112,0.278205,0.335566,0.048513
1,2016-02-07 14:00:00,0.320615,1.272727,0.090909,0.608696,1.000000,0.200000,0.076923,1,0.327568,0.320510,0.265606,0.334993,0.048556
2,2016-02-07 15:00:00,0.317244,1.272727,0.090909,0.652174,1.000000,0.200000,0.076923,1,0.320615,0.309913,0.257095,0.335000,0.048548
3,2016-02-07 16:00:00,0.323480,1.272727,0.090909,0.695652,1.000000,0.200000,0.076923,1,0.317244,0.306900,0.262256,0.335440,0.047735
4,2016-02-07 17:00:00,0.343811,1.272727,0.090909,0.739130,1.000000,0.200000,0.076923,1,0.323480,0.322638,0.283451,0.336435,0.045987
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21775,2018-08-02 20:00:00,0.621784,1.454545,0.636364,0.869565,0.500000,0.033333,0.576923,0,0.655156,0.681934,0.669820,0.628587,0.532395
21776,2018-08-02 21:00:00,0.604909,1.454545,0.636364,0.913043,0.500000,0.033333,0.576923,0,0.621784,0.662404,0.632003,0.624977,0.521400
21777,2018-08-02 22:00:00,0.569009,1.454545,0.636364,0.956522,0.500000,0.033333,0.576923,0,0.604909,0.622564,0.591889,0.621527,0.512173
21778,2018-08-02 23:00:00,0.504709,1.454545,0.636364,1.000000,0.500000,0.033333,0.576923,0,0.569009,0.550342,0.521058,0.618313,0.506551


In [50]:
df_val

,Datetime,PJME_MW,year,month,hour,day_num,day_of_month,week_of_year,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24
0,2013-08-14 01:00:00,0.319751,1.000000,0.636364,0.043478,0.333333,0.433333,0.615385,0,0.379374,0.349141,0.263647,0.522917,0.415302
1,2013-08-14 02:00:00,0.274055,1.000000,0.636364,0.086957,0.333333,0.433333,0.615385,0,0.319751,0.311767,0.234952,0.521153,0.422386
2,2013-08-14 03:00:00,0.243148,1.000000,0.636364,0.130435,0.333333,0.433333,0.615385,0,0.274055,0.290909,0.217950,0.518890,0.434119
3,2013-08-14 04:00:00,0.223891,1.000000,0.636364,0.173913,0.333333,0.433333,0.615385,0,0.243148,0.280733,0.211904,0.516024,0.450687
4,2013-08-14 05:00:00,0.218645,1.000000,0.636364,0.217391,0.333333,0.433333,0.615385,0,0.223891,0.289729,0.220752,0.512612,0.471038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21775,2016-02-07 08:00:00,0.347898,1.272727,0.090909,0.347826,1.000000,0.200000,0.076923,1,0.329864,0.381839,0.309891,0.344615,0.072727
21776,2016-02-07 09:00:00,0.359654,1.272727,0.090909,0.391304,1.000000,0.200000,0.076923,1,0.347898,0.398820,0.325103,0.342579,0.068262
21777,2016-02-07 10:00:00,0.355019,1.272727,0.090909,0.434783,1.000000,0.200000,0.076923,1,0.359654,0.388560,0.322996,0.340228,0.059683
21778,2016-02-07 11:00:00,0.344823,1.272727,0.090909,0.478261,1.000000,0.200000,0.076923,1,0.355019,0.371411,0.306268,0.338215,0.053128


In [ ]:
def create_sequences(df,seq_length, target_col='PJME_MW',horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [52]:
SEQ_LEN = 24
HORIZON = 1

X_train, y_train = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val, y_val     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test, y_test   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")

X_train Shape: (101616, 24, 13)
X_test Shape:  (21756, 24, 13)


In [53]:
'''BASIC RNN'''

'BASIC RNN'

In [54]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=20,batch_size=64,verbose=1)

Epoch 1/20


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0039 - mae: 0.0414 - val_loss: 4.7880e-04 - val_mae: 0.0169
Epoch 2/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 7.9970e-04 - mae: 0.0217 - val_loss: 3.3716e-04 - val_mae: 0.0141
Epoch 3/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 5.3370e-04 - mae: 0.0176 - val_loss: 2.2247e-04 - val_mae: 0.0118
Epoch 4/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 4.0683e-04 - mae: 0.0154 - val_loss: 1.7816e-04 - val_mae: 0.0102
Epoch 5/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 3.4129e-04 - mae: 0.0140 - val_loss: 1.8649e-04 - val_mae: 0.0110
Epoch 6/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 2.9389e-04 - mae: 0.0130 - val_loss: 9.8832e-05 - val_mae: 0.0077
Epoch 7/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 2.6939e-04 - mae: 0.0124 - val_loss: 8.1762e-05 - val_mae: 0.0068
Epoch 8/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 2.4077e-04 - mae: 0.0117 - val_loss: 7.2316e-05 - val_mae

In [55]:

y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))
y_test_mw = scaler.inverse_transform(y_test.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
MAE:  1337.95 MW
RMSE: 1639.15 MW
MAPE: 3.91%
R2:   0.9353


In [58]:
y_pred =  model_rnn.predict(X_test)

y_pred = scaler.inverse_transform(
    y_pred.reshape(-1, 1)
)

y_actual = scaler.inverse_transform(
    y_test.reshape(-1, 1)
)

print("Actual:", y_actual[:10].flatten())
print("Predicted:", y_pred[:10].flatten())

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Actual: [32732. 32456. 32341. 32717. 33832. 35835. 36756. 36332. 35503. 33966.]
Predicted: [31048.373 30853.797 31037.217 31412.875 32592.021 33979.59  34402.84
 33990.363 33712.645 32653.912]


In [59]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_train min:", X_train.min())
print("X_train max:", X_train.max())

print("y_train min:", y_train.min())
print("y_train max:", y_train.max())

print("First y_train:", y_train[:10])

X_train shape: (101616, 24, 13)
y_train shape: (101616,)
X_train min: 0.0
X_train max: 1.0000000000000002
y_train min: 0.0
y_train max: 1.0
First y_train: [0.26545876 0.2447909  0.23613189 0.2335405  0.24677131 0.29086695
 0.37770989 0.42656694 0.4183925  0.40790056]


In [60]:
print("Prediction min:", y_pred.min())
print("Prediction max:", y_pred.max())

Prediction min: 19555.406
Prediction max: 51811.8


In [ ]:
#lstm

In [61]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=20,batch_size=64,verbose=1)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0034 - mae: 0.0399 - val_loss: 5.5295e-04 - val_mae: 0.0186
Epoch 2/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 6.8536e-04 - mae: 0.0201 - val_loss: 2.2597e-04 - val_mae: 0.0118
Epoch 3/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 4.4716e-04 - mae: 0.0161 - val_loss: 2.8932e-04 - val_mae: 0.0135
Epoch 4/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 3.6016e-04 - mae: 0.0145 - val_loss: 2.2142e-04 - val_mae: 0.0112
Epoch 5/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 2.9617e-04 - mae: 0.0131 - val_loss: 5.5676e-04 - val_mae: 0.0194
Epoch 6/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 2.4947e-04 - mae: 0.0120 - val_loss: 4.2646e-04 - val_mae: 0.0160
Epoch 7/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 2.0294e-04 - mae: 0.0108 - val_loss: 5.2793e-04 - val_mae: 0.0173
Epoch 8/20
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 1.6946e-04 - mae: 0.0098 - val_

In [62]:
y_pred_lstm_scaled = model_lstm.predict(X_test)

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [63]:
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))

In [64]:
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

MAE:  2102.85 MW
RMSE: 2429.23 MW
MAPE: 6.29%
R2: 0.86%


In [ ]:
#GRU

In [65]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0025 - mae: 0.0349 - val_loss: 4.0708e-04 - val_mae: 0.0153
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 5.7179e-04 - mae: 0.0183 - val_loss: 6.8804e-04 - val_mae: 0.0213
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 3.9597e-04 - mae: 0.0152 - val_loss: 1.9931e-04 - val_mae: 0.0106
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 3.1499e-04 - mae: 0.0135 - val_loss: 2.1798e-04 - val_mae: 0.0111
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 2.5225e-04 - mae: 0.0120 - val_loss: 2.7086e-04 - val_mae: 0.0124
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 2.0113e-04 - mae: 0.0107 - val_loss: 6.8269e-04 - val_mae: 0.0213
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 1.6588e-04 - mae: 0.0097 - val_loss: 4.8571e-04 - val_mae: 0.0174
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 1.3873e-04 - mae: 0.0088 - val

In [66]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1449.35 MW
RMSE: 1823.33 MW
MAPE: 4.19%
R2:   0.9200


In [ ]:
#bidirectional lstm

In [68]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.0026 - mae: 0.0338 - val_loss: 3.4095e-04 - val_mae: 0.0140
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 26s 17ms/step - loss: 5.6565e-04 - mae: 0.0183 - val_loss: 2.4239e-04 - val_mae: 0.0119
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 3.5496e-04 - mae: 0.0144 - val_loss: 1.3795e-04 - val_mae: 0.0089
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 2.5905e-04 - mae: 0.0122 - val_loss: 1.8157e-04 - val_mae: 0.0107
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 2.2052e-04 - mae: 0.0112 - val_loss: 7.8194e-05 - val_mae: 0.0067
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 1.8974e-04 - mae: 0.0104 - val_loss: 8.2460e-05 - val_mae: 0.0071
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 1.6407e-04 - mae: 0.0097 - val_loss: 7.0750e-05 - val_mae: 0.0062
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 1.4141e-04 - mae: 0.0089 - val

In [69]:

y_pred_bilstm_scaled = model_bilstm.predict(X_test)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))


mae_bilstm = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  646.22 MW
RMSE: 754.40 MW
MAPE: 2.04%
R2:   0.9863


In [ ]:
'''48'''

In [86]:
SEQ_LEN = 48
HORIZON = 24

X_train48, y_train48 = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val48, y_val48     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test48, y_test48   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print("X_train:", X_train48.shape)
print("y_train:", y_train48.shape)

print("X_test:", X_test48.shape)
print("y_test:", y_test48.shape)

X_train: (101569, 48, 13)
y_train: (101569, 24)
X_test: (21709, 48, 13)
y_test: (21709, 24)


In [87]:
'''baseline cnn with 48'''

'baseline cnn with 48'

In [88]:
input_shape = (X_train48.shape[1], X_train48.shape[2])

# Build Basic RNN
model_rnn48 = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn48.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn48.fit(X_train48, y_train48,validation_data=(X_val48, y_val48),epochs=10,batch_size=64,verbose=1)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.0091 - mae: 0.0645 - val_loss: 0.0034 - val_mae: 0.0471
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0026 - mae: 0.0385 - val_loss: 0.0025 - val_mae: 0.0373
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0022 - mae: 0.0354 - val_loss: 0.0022 - val_mae: 0.0349
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0021 - mae: 0.0343 - val_loss: 0.0033 - val_mae: 0.0450
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0020 - mae: 0.0338 - val_loss: 0.0028 - val_mae: 0.0385
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0020 - mae: 0.0335 - val_loss: 0.0021 - val_mae: 0.0335
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0019 - mae: 0.0328 - val_loss: 0.0020 - val_mae: 0.0341
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0019 - mae: 0.0324 - val_loss: 0.0021 - val_mae: 0.0341
Epoch 9/10
1588/1588 ━━━━━━━━━━━━

In [89]:

y_pred_rnn_scaled = model_rnn48.predict(X_test48)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_rnn_scaled.reshape(-1, 1))
y_test_mw = scaler.inverse_transform(y_test48.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

679/679 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1524.32 MW
RMSE: 2102.19 MW
MAPE: 4.89%
R2:   0.8936


In [ ]:
'''lstm'''

In [90]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train48.shape[1], X_train48.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train48, y_train48,validation_data=(X_val48, y_val48),epochs=10,batch_size=64,verbose=1)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - loss: 0.0067 - mae: 0.0570 - val_loss: 0.0025 - val_mae: 0.0378
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - loss: 0.0024 - mae: 0.0371 - val_loss: 0.0022 - val_mae: 0.0342
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - loss: 0.0020 - mae: 0.0338 - val_loss: 0.0019 - val_mae: 0.0315
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - loss: 0.0018 - mae: 0.0320 - val_loss: 0.0022 - val_mae: 0.0341
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 30s 19ms/step - loss: 0.0018 - mae: 0.0311 - val_loss: 0.0018 - val_mae: 0.0307
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 30s 19ms/step - loss: 0.0017 - mae: 0.0305 - val_loss: 0.0020 - val_mae: 0.0350
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 30s 19ms/step - loss: 0.0017 - mae: 0.0300 - val_loss: 0.0016 - val_mae: 0.0286
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - loss: 0.0016 - mae: 0.0294 - val_loss: 0.0016 - val_mae: 0.0284
Epoch 9/10
1588/1588 ━━━

In [91]:
y_pred_lstm_scaled = model_lstm.predict(X_test48)

y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

679/679 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1435.23 MW
RMSE: 1977.20 MW
MAPE: 4.60%
R2:   0.9059


In [84]:
'''GRU'''

'GRU'

In [92]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train48.shape[1], X_train48.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train48, y_train48,
    validation_data=(X_val48, y_val48),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 37s 22ms/step - loss: 0.0070 - mae: 0.0584 - val_loss: 0.0025 - val_mae: 0.0378
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 39s 25ms/step - loss: 0.0022 - mae: 0.0349 - val_loss: 0.0018 - val_mae: 0.0311
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 38s 24ms/step - loss: 0.0019 - mae: 0.0322 - val_loss: 0.0020 - val_mae: 0.0329
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 37s 23ms/step - loss: 0.0018 - mae: 0.0313 - val_loss: 0.0019 - val_mae: 0.0323
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 37s 23ms/step - loss: 0.0017 - mae: 0.0306 - val_loss: 0.0017 - val_mae: 0.0297
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 36s 22ms/step - loss: 0.0017 - mae: 0.0301 - val_loss: 0.0016 - val_mae: 0.0289
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.0016 - mae: 0.0297 - val_loss: 0.0017 - val_mae: 0.0310
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 36s 22ms/step - loss: 0.0016 - mae: 0.0293 - val_loss: 0.0015 - val_mae: 0.0279
Epoch 9/10
1588/1588 ━━━

In [93]:
y_pred_GRU_scaled = model_gru.predict(X_test48)

y_pred_GRU_mw = scaler.inverse_transform(y_pred_GRU_scaled.reshape(-1, 1))

mae_GRU  = mean_absolute_error(y_test_mw, y_pred_GRU_mw)
rmse_GRU = np.sqrt(mean_squared_error(y_test_mw, y_pred_GRU_mw))
mape_GRU = np.mean(np.abs((y_test_mw - y_pred_GRU_mw) / y_test_mw)) * 100
r2_GRU  = r2_score(y_test_mw, y_pred_GRU_mw)

print(f"MAE:  {mae_GRU:.2f} MW")
print(f"RMSE: {rmse_GRU:.2f} MW")
print(f"MAPE: {mape_GRU:.2f}%")
print(f"R2:   {r2_GRU:.4f}")

679/679 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1499.02 MW
RMSE: 2041.41 MW
MAPE: 4.77%
R2:   0.8997


In [ ]:
'''bidirectional'''

In [94]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train48.shape[1], X_train48.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train48, y_train48,
    validation_data=(X_val48, y_val48),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 37s 21ms/step - loss: 0.0056 - mae: 0.0522 - val_loss: 0.0034 - val_mae: 0.0470
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0021 - mae: 0.0341 - val_loss: 0.0019 - val_mae: 0.0316
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 56s 35ms/step - loss: 0.0018 - mae: 0.0312 - val_loss: 0.0018 - val_mae: 0.0309
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 48s 30ms/step - loss: 0.0017 - mae: 0.0302 - val_loss: 0.0020 - val_mae: 0.0323
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 44s 28ms/step - loss: 0.0016 - mae: 0.0293 - val_loss: 0.0016 - val_mae: 0.0283
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 39s 24ms/step - loss: 0.0015 - mae: 0.0286 - val_loss: 0.0015 - val_mae: 0.0282
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 39s 25ms/step - loss: 0.0015 - mae: 0.0281 - val_loss: 0.0018 - val_mae: 0.0310
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 39s 24ms/step - loss: 0.0015 - mae: 0.0278 - val_loss: 0.0015 - val_mae: 0.0280
Epoch 9/10
1588/1588 ━━━

In [95]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test48)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

679/679 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
MAE:  1590.67 MW
RMSE: 2170.30 MW
MAPE: 5.07%
R2:   0.8866


In [ ]:
'''168'''

In [96]:
SEQ_LEN = 168
HORIZON = 24

X_train168, y_train168 = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val168, y_val168     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test168, y_test168   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print("X_train:", X_train168.shape)
print("y_train:", y_train168.shape)

print("X_test:", X_test168.shape)
print("y_test:", y_test168.shape)

X_train: (101449, 168, 13)
y_train: (101449, 24)
X_test: (21589, 168, 13)
y_test: (21589, 24)


In [99]:
y_test_mw = scaler.inverse_transform(y_test168.reshape(-1, 1))

In [ ]:
'''basline rnn'''

In [97]:
input_shape = (X_train168.shape[1], X_train168.shape[2])

#Basic RNN
model_rnn168 = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn168.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn168.fit(X_train168, y_train168,validation_data=(X_val168, y_val168),epochs=10,batch_size=64,verbose=1)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 35s 21ms/step - loss: 0.0093 - mae: 0.0658 - val_loss: 0.0027 - val_mae: 0.0391
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 36s 22ms/step - loss: 0.0029 - mae: 0.0406 - val_loss: 0.0022 - val_mae: 0.0351
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 37s 23ms/step - loss: 0.0024 - mae: 0.0370 - val_loss: 0.0020 - val_mae: 0.0331
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - loss: 0.0022 - mae: 0.0351 - val_loss: 0.0020 - val_mae: 0.0324
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 32s 20ms/step - loss: 0.0021 - mae: 0.0343 - val_loss: 0.0019 - val_mae: 0.0320
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 36s 22ms/step - loss: 0.0020 - mae: 0.0338 - val_loss: 0.0019 - val_mae: 0.0319
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 34s 22ms/step - loss: 0.0020 - mae: 0.0334 - val_loss: 0.0018 - val_mae: 0.0310
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 32s 20ms/step - loss: 0.0020 - mae: 0.0330 - val_loss: 0.0018 - val_mae: 0.0303
Epoch 9/10
1586/1586 ━━━

In [100]:
y_pred_rnn_scaled = model_rnn168.predict(X_test168)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_rnn_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step
MAE:  1564.34 MW
RMSE: 2131.29 MW
MAPE: 4.99%
R2:   0.8905


In [ ]:
'''lstm'''

In [101]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train168, y_train168,validation_data=(X_val168, y_val168),epochs=10,batch_size=64,verbose=1)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 104s 64ms/step - loss: 0.0066 - mae: 0.0572 - val_loss: 0.0026 - val_mae: 0.0382
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 120s 76ms/step - loss: 0.0024 - mae: 0.0368 - val_loss: 0.0019 - val_mae: 0.0322
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 122s 77ms/step - loss: 0.0020 - mae: 0.0332 - val_loss: 0.0019 - val_mae: 0.0321
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 121s 76ms/step - loss: 0.0018 - mae: 0.0318 - val_loss: 0.0018 - val_mae: 0.0308
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 122s 77ms/step - loss: 0.0017 - mae: 0.0309 - val_loss: 0.0018 - val_mae: 0.0302
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 121s 77ms/step - loss: 0.0017 - mae: 0.0303 - val_loss: 0.0016 - val_mae: 0.0286
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 102s 64ms/step - loss: 0.0016 - mae: 0.0297 - val_loss: 0.0017 - val_mae: 0.0296
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 100s 63ms/step - loss: 0.0016 - mae: 0.0293 - val_loss: 0.0018 - val_mae: 0.0303
Epoch 9/10
1586/

In [ ]:
y_pred_lstm_scaled = model_lstm.predict(X_test168)

y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))

mae_lstm  = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm  = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} ")
print(f"RMSE: {rmse_lstm:.2f} ")
print(f"MAPE: {mape_lstm:.2f}")
print(f"R2:   {r2_lstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step
MAE:  1434.24 MW
RMSE: 2013.61 MW
MAPE: 4.48%
R2:   0.9022


In [ ]:
'''GRU'''

In [103]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 157s 97ms/step - loss: 0.0072 - mae: 0.0594 - val_loss: 0.0024 - val_mae: 0.0374
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 157s 99ms/step - loss: 0.0022 - mae: 0.0354 - val_loss: 0.0019 - val_mae: 0.0317
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 161s 102ms/step - loss: 0.0019 - mae: 0.0324 - val_loss: 0.0018 - val_mae: 0.0310
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 165s 104ms/step - loss: 0.0018 - mae: 0.0314 - val_loss: 0.0017 - val_mae: 0.0298
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 154s 97ms/step - loss: 0.0017 - mae: 0.0305 - val_loss: 0.0017 - val_mae: 0.0296
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 129s 82ms/step - loss: 0.0017 - mae: 0.0300 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 121s 76ms/step - loss: 0.0016 - mae: 0.0297 - val_loss: 0.0015 - val_mae: 0.0282
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 145s 91ms/step - loss: 0.0016 - mae: 0.0292 - val_loss: 0.0015 - val_mae: 0.0277
Epoch 9/10
158

In [104]:
y_pred_GRU_scaled = model_gru.predict(X_test168)

y_pred_GRU_mw = scaler.inverse_transform(y_pred_GRU_scaled.reshape(-1, 1))

mae_GRU  = mean_absolute_error(y_test_mw, y_pred_GRU_mw)
rmse_GRU = np.sqrt(mean_squared_error(y_test_mw, y_pred_GRU_mw))
mape_GRU = np.mean(np.abs((y_test_mw - y_pred_GRU_mw) / y_test_mw)) * 100
r2_GRU  = r2_score(y_test_mw, y_pred_GRU_mw)

print(f"MAE:  {mae_GRU:.2f} MW")
print(f"RMSE: {rmse_GRU:.2f} MW")
print(f"MAPE: {mape_GRU:.2f}%")
print(f"R2:   {r2_GRU:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step
MAE:  1502.94 MW
RMSE: 2077.62 MW
MAPE: 4.66%
R2:   0.8959


In [ ]:
'''bilstm'''

In [105]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 162s 101ms/step - loss: 0.0059 - mae: 0.0537 - val_loss: 0.0023 - val_mae: 0.0351
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 167s 105ms/step - loss: 0.0021 - mae: 0.0342 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 167s 105ms/step - loss: 0.0018 - mae: 0.0314 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 161s 102ms/step - loss: 0.0017 - mae: 0.0302 - val_loss: 0.0016 - val_mae: 0.0289
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 161s 101ms/step - loss: 0.0016 - mae: 0.0294 - val_loss: 0.0016 - val_mae: 0.0284
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 165s 104ms/step - loss: 0.0015 - mae: 0.0288 - val_loss: 0.0016 - val_mae: 0.0286
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 129s 81ms/step - loss: 0.0015 - mae: 0.0281 - val_loss: 0.0015 - val_mae: 0.0274
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 154s 97ms/step - loss: 0.0014 - mae: 0.0278 - val_loss: 0.0014 - val_mae: 0.0268
Epoch 9/10

In [106]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step
MAE:  1405.28 MW
RMSE: 1985.54 MW
MAPE: 4.37%
R2:   0.9050
